# Reproduce the 50-image bottle fine-tuning experiment

This is an **unexecuted Colab template**. Recorded local results are in the repository's `results/experiment2/` and report, not in this notebook's empty outputs. Choose a GPU runtime before running the cells.

Training uses YOLO11m with its first 10 layers frozen, ordinary fine-tuning rather than LoRA. The bundled dataset is a fixed 40-image training / 10-image validation course split sampled from Open Images. The validation split also selects the checkpoint, so it is not an independent test set. The model learns one class, bottle. The public repository contains the original apartment photographs and dataset attribution; no new photo upload is required.

Course reference: [Roboflow YOLO11 fine-tuning tutorial](https://colab.research.google.com/github/roboflow-ai/notebooks/blob/main/notebooks/train-yolo11-object-detection-on-custom-dataset.ipynb).


In [ ]:
%pip install -q ultralytics==8.4.142


In [ ]:
from pathlib import Path
import subprocess
import sys
import torch

project = Path('/content/CSE498')
if not project.exists():
    subprocess.run(['git', 'clone', 'https://github.com/lllingshen/CSE498.git', str(project)], check=True)
assert torch.cuda.is_available(), 'Select a GPU runtime in Colab first.'
print('Using the existing Colab PyTorch:', torch.__version__)
print('GPU:', torch.cuda.get_device_name(0))
subprocess.run([sys.executable, str(project / 'setup_models.py'), '--models', 'yolo11m.pt'], check=True)
subprocess.run([sys.executable, str(project / 'src/prepare_dataset.py')], check=True)


## Train

The script resolves dataset paths to this Colab checkout and writes new outputs under `runs/`. Defaults match the local experiment: 30 epochs maximum, patience 10, image size 960, batch 4, freeze 10, AdamW, learning rate 0.0005, seed 42, workers 0, AMP disabled. Different hardware and PyTorch versions can produce different numbers.

The bundled `models/best_bottles.pt` remains the recorded local checkpoint. The new training run does not overwrite it.


In [ ]:
subprocess.run([sys.executable, str(project / 'src/train_bottles.py'),
                '--device', '0', '--output', str(project / 'runs/colab_training')], check=True)
candidates = list((project / 'runs').glob('colab_training*/weights/best.pt'))
best_path = max(candidates, key=lambda path: path.stat().st_mtime)
run_dir = best_path.parent.parent
print('New Colab checkpoint:', best_path)


In [ ]:
from IPython.display import display, Image
import pandas as pd

display(Image(filename=str(run_dir / 'results.png')))
display(pd.read_csv(run_dir / 'results.csv').tail())


## Compare the new checkpoint and download it

The comparison below explicitly uses the newly trained checkpoint. It evaluates only bottle scores, maps original COCO class 39 and fine-tuned class 0 consistently, and reruns the two apartment photos with the same inference settings. AP is measured on the small validation set; apartment box counts are not accuracy. Examine false positives, missed bottles, and duplicate boxes.

The apartment photos were not used for training or checkpoint selection. More boxes or a higher validation score do not establish improvement for all household objects.


In [ ]:
comparison = project / 'runs/colab_comparison'
subprocess.run([sys.executable, str(project / 'src/compare_bottles.py'),
                '--weights', str(best_path), '--output', str(comparison)], check=True)
import json
print(json.loads((comparison / 'metrics.json').read_text()))
for name in ['before_1P9A1048.jpg', 'after_1P9A1048.jpg',
             'before_1P9A1049.jpg', 'after_1P9A1049.jpg']:
    print(name)
    display(Image(filename=str(comparison / name), width=900))
from google.colab import files
files.download(str(best_path))
